In [1]:
%useLatestDescriptors
%use dataframe
%use kandy

In [135]:
USE{
    dependencies("org.json:json:20250107")
}

In [136]:
import org.json.XML

In [107]:
enum class SeaSector(val simpleName: String) {
    EAST("001"),
    WEST("002"),
    SOUTH("003")
}

In [146]:
val serviceKeyFilePath = "/Volumes/WorkSpace/Notebook/data/OceanMensurationService.json"
val SDATE = "20250225"
val EDATE = "20250225"
val numOfRows = "100"
val maxPage = 100

In [147]:
fun load(path:String, maxPage:Int): AnyFrame {
    val rows = mutableListOf<AnyFrame>()
    var requestPage = 1
    do{
        val pagePath = "$path&pageNo=$requestPage"
        val jsonData = XML.toJSONObject(DataFrame.read(pagePath).toString())
        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
        try {
            val instanceDf = df.get("response").get("body").get("items").get("item").toDataFrame()
            requestPage += 1
            rows.add(instanceDf)
        } catch(e: Exception) {
            print(e.localizedMessage)
            break
        }
    } while (rows.size < maxPage )
    return rows.concat()
}

In [169]:
val serviceInfo = DataRow.readJson(path=serviceKeyFilePath)

In [171]:
val url = "${serviceInfo.endpoint_apis}/${serviceInfo.list_apis}?ServiceKey=${serviceInfo.key_apis}&GRU_NAM=${SeaSector.EAST.simpleName}&SDATE=$SDATE&EDATE=$EDATE&numOfRows=$numOfRows"

In [150]:
val rawDf = load(url,maxPage )

Can not get nested column 'item' from ValueColumn 'items'

In [151]:
val rawDfConcat = rawDf.item.concat()
rawDfConcat

staNamKor,cdt_1,obsDtm,staCde,obsTim,wtrTmp_1,gruNam,wtrTmp_3,wtrTmp_2
덕천,34.351000,2025-02-25 23:30:00,bdch3,233000,10.600000,동해,null,null
구룡포 하정,null,2025-02-25 23:30:00,fghe8,233000,10.200000,동해,null,null
고성 가진,null,2025-02-25 23:30:00,fggo3,233000,5.300000,동해,5,5.200000
온양,34.355000,2025-02-25 23:30:00,byyh3,233000,10.100000,동해,null,null
양양,null,2025-02-25 23:30:00,byy87,233000,5.400000,동해,5.400000,5.400000
영덕,null,2025-02-25 23:30:00,byd8a,233000,10.100000,동해,10.100000,10
삼척,null,2025-02-25 23:30:00,bsc87,233000,9.600000,동해,8.800000,8.900000
나곡,34.318000,2025-02-25 23:30:00,bngh3,233000,10.200000,동해,null,null
진하,34.344000,2025-02-25 23:30:00,bjhh3,233000,12.200000,동해,null,null
고리,35.234000,2025-02-25 23:30:00,bgrh3,233000,12.200000,동해,null,null


In [152]:
val rawDfParse = rawDfConcat.parse()

In [153]:
rawDfParse

staNamKor,cdt_1,obsDtm,staCde,obsTim,wtrTmp_1,gruNam,wtrTmp_3,wtrTmp_2
덕천,34.351000,2025-02-25T23:30,bdch3,233000,10.600000,동해,null,null
구룡포 하정,null,2025-02-25T23:30,fghe8,233000,10.200000,동해,null,null
고성 가진,null,2025-02-25T23:30,fggo3,233000,5.300000,동해,5,5.200000
온양,34.355000,2025-02-25T23:30,byyh3,233000,10.100000,동해,null,null
양양,null,2025-02-25T23:30,byy87,233000,5.400000,동해,5.400000,5.400000
영덕,null,2025-02-25T23:30,byd8a,233000,10.100000,동해,10.100000,10
삼척,null,2025-02-25T23:30,bsc87,233000,9.600000,동해,8.800000,8.900000
나곡,34.318000,2025-02-25T23:30,bngh3,233000,10.200000,동해,null,null
진하,34.344000,2025-02-25T23:30,bjhh3,233000,12.200000,동해,null,null
고리,35.234000,2025-02-25T23:30,bgrh3,233000,12.200000,동해,null,null


In [154]:
rawDfParse.staCde.distinct()

staCde
bdch3
fghe8
fggo3
byyh3
byy87
byd8a
bsc87
bngh3
bjhh3
bgrh3


In [155]:
rawDfParse.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staNamKor,String,572,13,0,강릉,51,null,null,강릉,나곡,진하
cdt_1,Double?,572,119,334,34.341000,6,34.506214,0.330440,34.259000,34.347000,35.326000
obsDtm,kotlinx.datetime.LocalDateTime,572,48,0,2025-02-25T23:30,13,null,null,2025-02-25T00:00,2025-02-25T12:00,2025-02-25T23:30
staCde,String,572,13,0,bgna3,51,null,null,bdch3,bngh3,fghe8
obsTim,Comparable<*>,572,48,0,233000,13,null,null,null,null,null
wtrTmp_1,Number,572,55,0,5.400000,57,9.713112,2.442327,5.300000,10.300000,13.600000
gruNam,String,572,1,0,동해,572,null,null,동해,동해,동해
wtrTmp_3,Number?,572,28,283,6.400000,34,8.062976,2.583087,5.000000,8.300000,12.300000
wtrTmp_2,Number?,572,24,283,5.300000,51,8.131834,2.546189,5.200000,8.900000,12.300000


In [156]:
rawDfParse.plot{
    x(obsDtm) { axis.name = "관측일시"}
    y(wtrTmp_1) {axis.name ="표층수온"}
    y.axis.limits = 0.0..15.0
    line{
        color(staNamKor){
            scale = continuous(Color.GREEN..Color.RED)
            legend{
                name = "관측소명"
            }
        }
    }

    layout {
        title = "관측지점별 해수 정보"
        size = 1200 to 500
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="iabevT"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"관측지점별 해수 정보"
},
"mapping":{
},
"coord":{
"name":"cartesian",
"flip":false,
"ylim":[0.0,15.0]
},
"data":{
"staNamKor":["덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","양양","영덕","삼척","나곡","고리","강릉","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","온양","양양","영덕","삼척","강릉","기장(한수원)","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","고리","강릉","기장(한수원)","기장","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","양양","영덕","삼척","나곡","강릉","기장(한수원)","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","양양","영덕","삼척","진하","강릉","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","양양","영덕","삼척","나곡","진하","고리","강릉","기장","고리","강릉","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","강릉","기장(한수원)","기장","덕천","구룡포 하정","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","양양","영덕","삼척","진하","강릉","기장(한수원)","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","양양","영덕","삼척","강릉","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","양양","영덕","삼척","고리","강릉","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","양양","영덕","삼척","나곡","강릉","기장","강릉","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","온양","양양","영덕","삼척","진하","강릉","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","기장","구룡포 하정","고성 가진","양양","영덕","삼척","나곡","강릉","덕천","구룡포 하정","고성 가진","온양","양양","영덕","삼척","나곡","진하","고리","강릉","기장(한수원)","기장","덕천","기장","기장(한수원)","강릉","고리","진하","나곡","삼척","영덕","양양","온양","고성 가진","구룡포 하정"],
"obsDtm":[1.7405262E12,1.7405262E12,1.7405262E12,1.7405262E12,1.740

In [157]:
val staCodeDf = DataFrame.readCSV("/Volumes/WorkSpace/Notebook/data/STA_CODE_NEW2.csv")
staCodeDf

staCde,gruNam,name,longitude,latitude
fgmk6,남해,강진 마량,126.813286,34.444798
fgsl6,남해,강진 사초,126.769100,34.459900
ms002,남해,거제,128.458900,34.535900
fgg4c,남해,거제 가배,128.566400,34.785100
fggj7,남해,거제 가조,128.514200,34.977200
gi086,남해,거제 일운,128.709400,34.803800
btei5,남해,거제 해금강,128.690833,34.735833
bgyga,남해,고성 용정,128.500860,34.989700
bgjgb,남해,고성 장좌,128.465450,34.970800
ys002,남해,고흥,127.350900,34.261200


In [158]:
val df = rawDfParse.groupBy{staCde}.aggregate {
    val currentData = maxBy{obsDtm}
    currentData.obsDtm into "datetime"
    currentData.gruNam into "gruNam"
    currentData.staNamKor into "name"
    currentData.wtrTmp_1 into "currentTemp"
}.join(staCodeDf.select{  gruNam and  name and staCde and latitude and longitude}){
     "gruNam" and "name" and "staCde"
}

df

staCde,datetime,gruNam,name,currentTemp,latitude,longitude
bdch3,2025-02-25T23:30,동해,덕천,10.600000,37.100000,129.404100
fghe8,2025-02-25T23:30,동해,구룡포 하정,10.200000,35.960700,129.549700
fggo3,2025-02-25T23:30,동해,고성 가진,5.300000,38.368100,128.523900
byyh3,2025-02-25T23:30,동해,온양,10.100000,37.019400,129.425000
byy87,2025-02-25T23:30,동해,양양,5.400000,38.080800,128.699800
byd8a,2025-02-25T23:30,동해,영덕,10.100000,36.573700,129.437000
bsc87,2025-02-25T23:30,동해,삼척,9.600000,37.302300,129.312700
bngh3,2025-02-25T23:30,동해,나곡,10.200000,37.119100,129.395800
bjhh3,2025-02-25T23:30,동해,진하,12.200000,35.384700,129.368000
bgrh3,2025-02-25T23:30,동해,고리,12.200000,35.318600,129.314700


In [159]:
df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staCde,String,13,13,0,bdch3,1,null,null,bdch3,bngh3,fghe8
datetime,kotlinx.datetime.LocalDateTime,13,1,0,2025-02-25T23:30,13,null,null,2025-02-25T23:30,2025-02-25T23:30,2025-02-25T23:30
gruNam,String,13,1,0,동해,13,null,null,동해,동해,동해
name,String,13,13,0,덕천,1,null,null,강릉,나곡,진하
currentTemp,Number,13,10,0,10.200000,2,9.730769,2.416742,5.300000,10.200000,12.200000
latitude,Double,13,13,0,37.100000,1,36.645838,1.134755,35.182500,37.019400,38.368100
longitude,Double,13,13,0,129.404100,1,129.218623,0.306979,128.523900,129.314700,129.549700


In [160]:
val dfBoundary = df.add("tempBoundary"){
    when(currentTemp.toDouble()) {
        in 0.0..4.9 -> "Low"
        in 5.0..7.9 -> "Middle"
        in 8.0..15.9 -> "High"
        else -> "Unknown"
    }
}
dfBoundary

staCde,datetime,gruNam,name,currentTemp,latitude,longitude,tempBoundary
bdch3,2025-02-25T23:30,동해,덕천,10.600000,37.100000,129.404100,High
fghe8,2025-02-25T23:30,동해,구룡포 하정,10.200000,35.960700,129.549700,High
fggo3,2025-02-25T23:30,동해,고성 가진,5.300000,38.368100,128.523900,Middle
byyh3,2025-02-25T23:30,동해,온양,10.100000,37.019400,129.425000,High
byy87,2025-02-25T23:30,동해,양양,5.400000,38.080800,128.699800,Middle
byd8a,2025-02-25T23:30,동해,영덕,10.100000,36.573700,129.437000,High
bsc87,2025-02-25T23:30,동해,삼척,9.600000,37.302300,129.312700,High
bngh3,2025-02-25T23:30,동해,나곡,10.200000,37.119100,129.395800,High
bjhh3,2025-02-25T23:30,동해,진하,12.200000,35.384700,129.368000,High
bgrh3,2025-02-25T23:30,동해,고리,12.200000,35.318600,129.314700,High


In [161]:
dfBoundary.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
staCde,String,13,13,0,bdch3,1,null,null,bdch3,bngh3,fghe8
datetime,kotlinx.datetime.LocalDateTime,13,1,0,2025-02-25T23:30,13,null,null,2025-02-25T23:30,2025-02-25T23:30,2025-02-25T23:30
gruNam,String,13,1,0,동해,13,null,null,동해,동해,동해
name,String,13,13,0,덕천,1,null,null,강릉,나곡,진하
currentTemp,Number,13,10,0,10.200000,2,9.730769,2.416742,5.300000,10.200000,12.200000
latitude,Double,13,13,0,37.100000,1,36.645838,1.134755,35.182500,37.019400,38.368100
longitude,Double,13,13,0,129.404100,1,129.218623,0.306979,128.523900,129.314700,129.549700
tempBoundary,String,13,2,0,High,10,null,null,High,High,Middle


In [162]:
%useLatestDescriptors
%use lets-plot
%use lets-plot-gt

In [163]:
import org.geotools.geojson.feature.FeatureJSON
val southKr = FeatureJSON().readFeatureCollection(File("/Users/unchil/Downloads/southkorea.geojson").readText())
val mapBound = southKr.bounds
mapBound.expandBy(0.2)

In [164]:
fun<T> makePointGeoJSON( coordinates:List<Pair<Double, Double>>, properties:Map<String, List<T>>): String {
    val first_str = "{" + "\n" +
            "\t\"type\": \"FeatureCollection\"," + "\n" +
            "\t\"features\": [" + "\n"

    val end_str = "\t]" + "\n" +
            "}" + "\n"

    var features_str = ""

    coordinates.forEachIndexed { index,    pair ->

        features_str += "\t\t{\n" +
                "\t\t\t\"type\": \"Feature\",\n" +
                "\t\t\t\"geometry\": {\n" +
                "\t\t\t\t\"type\": \"Point\",\n" +
                "\t\t\t\t\"coordinates\": [${pair.first}, ${pair.second}]\n" +
                "\t\t\t},\n" +
                "\t\t\t\"properties\": {\n"

        properties.forEach { key, values ->
            features_str += "\t\t\t\t\"${key}\": \"${values[index]}\",\n"
        }

        features_str +=  "\t\t\t}\n" + "\t\t},\n"

    }

    return first_str + features_str + end_str
}

makePointGeoJSON(
    dfBoundary.select{longitude and latitude}.map{ Pair(longitude, latitude) }.toList(),
    dfBoundary.select{name and currentTemp and datetime and staCde and tempBoundary }.toMap()
)

{
	"type": "FeatureCollection",
	"features": [
		{
			"type": "Feature",
			"geometry": {
				"type": "Point",
				"coordinates": [129.4041, 37.1]
			},
			"properties": {
				"name": "덕천",
				"currentTemp": "10.6",
				"datetime": "2025-02-25T23:30",
				"staCde": "bdch3",
				"tempBoundary": "High",
			}
		},
		{
			"type": "Feature",
			"geometry": {
				"type": "Point",
				"coordinates": [129.5497, 35.9607]
			},
			"properties": {
				"name": "구룡포 하정",
				"currentTemp": "10.2",
				"datetime": "2025-02-25T23:30",
				"staCde": "fghe8",
				"tempBoundary": "High",
			}
		},
		{
			"type": "Feature",
			"geometry": {
				"type": "Point",
				"coordinates": [128.5239, 38.3681]
			},
			"properties": {
				"name": "고성 가진",
				"currentTemp": "5.3",
				"datetime": "2025-02-25T23:30",
				"staCde": "fggo3",
				"tempBoundary": "Middle",
			}
		},
		{
			"type": "Feature",
			"geometry": {
				"type": "Point",
				"coordinates": [129.425, 37.0194]
			},
			"properties": {
				"name":

In [165]:
val point = FeatureJSON().readFeatureCollection(
    makePointGeoJSON(
        dfBoundary.select{longitude and latitude}.map{Pair(longitude, latitude) }.toList(),
        dfBoundary.select{name and currentTemp and datetime and staCde and tempBoundary}.toMap()
    )
)

In [166]:
letsPlot() +
        geomMap(map = southKr.toSpatialDataset() , color = "#a1d99b", fill = "#e5f5e0" ) +
        geomRect(xmin= mapBound.minX, xmax =mapBound.maxX, ymax=mapBound.maxY, ymin= mapBound.minY, alpha = 0, fill = "black", color="#EFC623", size=6) +
        scaleXContinuous(expand = listOf(0.0, 0.0)) +
        scaleYContinuous(expand = listOf(0.0, 0.0)) +
        geomPoint(
            map= point.toSpatialDataset() ,
            showLegend=true,
            size=3,
            shape = 21,
            fillBy = "paint_a",
            tooltips = layerTooltips().title("@name[@staCde]").line("수온:@currentTemp°C\n@datetime")
        ){
            paint_a = "tempBoundary"
        } +
        geomText(map=point.toSpatialDataset() , position = positionNudge(x = +.2) , color="yellow"){ label =  "currentTemp"} +
        geomText(map=point.toSpatialDataset() , position = positionNudge(x = -.2) , color="black"){ label = "name" } +
        theme(panelBackground = elementRect(color = "black", fill = "#537ab5")) +
        ggtitle("Korea EastSea Water Quality") +
        ggsize( width = 1400, height = 1400)